# Reasoner on the five fixtures (Phase 2)

Runs the reasoner (`prompts/reasoner_v1.md`, with the whole rulebook in the prompt, DEC-19) on the hand-written fixtures and saves every prompt, reply and recommendation. The logic is in `src/strokecopilot/nlp/reasoner/`. Fixtures are synthetic, so nothing here touches patient data.

**Two modes**, set in section 1:
- `MODE = 'dummy'` (EXP-8): fixture fx02 twice. It checks that the API accepts the output schema, that the reply passes `schemas/recommendation.json`, and that the second call reads the cached system prompt. It also measures tokens and cost per call.
- `MODE = 'run'` (EXP-9): all five fixtures, twice each (10 calls), to see the decisions and how much they vary between runs.

**Before you start**
- Use a CPU runtime: Runtime → Change runtime type → None.
- In Secrets (the key icon), `ANTHROPIC_API_KEY` must have notebook access on.

**Run it:** Runtime → Run all, and allow Drive access when asked. Allow about 5 minutes for the dummy and 20 for the run. These are estimates: each call should take 30–90 seconds, which is unmeasured.

**Cost (estimate, unmeasured):** with `claude-sonnet-5-5`, about $0.10 per call, so about $0.20 for the dummy and $1 for the run. Most of it is output and thinking tokens. The dummy measures the real figure.

**Send back:** the `OK:` line from section 5, and any error in full. Everything is saved to `runs/EXP-8_reasoner_dummy/` (or `EXP-9_reasoner_fixtures/`) on Drive, where Claude can read it.

## 1. Settings

In [ ]:
MODE = 'dummy'               # 'dummy' (EXP-8) first, then 'run' (EXP-9)
MODEL = 'claude-sonnet-5-5'
EFFORT = 'high'              # the model's default
ROOT = '/content/drive/MyDrive/StrokeCopilot'
REPO_REF = 'main'
SETTINGS = {
    'dummy': {'experiment': 'EXP-8', 'name': 'EXP-8_reasoner_dummy', 'stems': ['fx02_thrombectomy_large_mismatch'], 'repeats': 2},
    'run': {'experiment': 'EXP-9', 'name': 'EXP-9_reasoner_fixtures', 'stems': None, 'repeats': 2},
}[MODE]
RUN_DIR = f"{ROOT}/runs/{SETTINGS['name']}"
LOCAL_DIR = f"/content/{SETTINGS['name']}"
print(f"{SETTINGS['experiment']} ({MODE}): {MODEL}, effort {EFFORT}.")

## 2. Install the package and mount Drive
The package is pinned to one commit, which the run records. Neither key is printed or kept in a variable.

In [ ]:
import json, os, shutil, subprocess, sys
from google.colab import drive, userdata

if any(m == 'strokecopilot' or m.startswith('strokecopilot.') for m in sys.modules):
    raise RuntimeError('This session already loaded an older copy of the package. Use Runtime → Restart session, then Run all.')

repo = 'https://github.com/parsa-hke/strokecopilot.git'
found = subprocess.run(['git', 'ls-remote', repo, REPO_REF], capture_output=True, text=True)
if found.returncode or not found.stdout.strip():
    raise RuntimeError(('Could not read the repo. ' + found.stderr)[-800:])
COMMIT = found.stdout.split()[0]
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'strokecopilot[llm] @ git+{repo}@{COMMIT}'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-800:])
# pip treats the same version as already installed, so the pinned commit is forced in
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'strokecopilot @ git+{repo}@{COMMIT}'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-800:])
del repo, found, result
print(f'Package installed at commit {COMMIT[:7]}.')

try:
    has_key = bool(userdata.get('ANTHROPIC_API_KEY'))
except Exception:
    has_key = False
if not has_key:
    raise RuntimeError('ANTHROPIC_API_KEY is not in Secrets, or notebook access is off.')
del has_key

drive.mount('/content/drive')
if os.path.exists(os.path.join(RUN_DIR, 'metrics.json')):
    raise RuntimeError(f'{RUN_DIR} already holds a finished run. Rename it on Drive, or change the experiment, before running again.')

## 3. Run the reasoner
One call per fixture and repeat. A refusal, a reply that is not JSON, or a reply that fails the schema is recorded as a failure and the run continues.

In [ ]:
from strokecopilot.nlp.reasoner import providers
from strokecopilot.nlp.reasoner import run as rr

llm = providers.ClaudeJSON(MODEL, effort=EFFORT, max_tokens=rr.REASONER_MAX_TOKENS)
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
metrics = rr.run_fixtures(llm, LOCAL_DIR, stems=SETTINGS['stems'], repeats=SETTINGS['repeats'],
                          extra_config={'experiment': SETTINGS['experiment'], 'mode': MODE, 'commit': COMMIT})

## 4. Per-call usage
The first call writes the system prompt to the cache; later calls within 5 minutes should read it.

In [ ]:
calls = []
for row in metrics['per_call']:
    record = json.load(open(f"{LOCAL_DIR}/calls/{row['fixture']}_r{row['repeat']}.json"))
    u = record['usage'] or {}
    calls.append(u)
    print(f"{row['fixture'][:4]} r{row['repeat']}: in {u.get('input_tokens', 0)}, cache write {u.get('cache_creation_input_tokens', 0)}, "
          f"cache read {u.get('cache_read_input_tokens', 0)}, out {u.get('output_tokens', 0)}, {row['seconds']} s")
cache_ok = len(calls) > 1 and all(u.get('cache_read_input_tokens', 0) > 0 for u in calls[1:])
print('Cache read on every call after the first:', cache_ok)

## 5. Save to Drive

In [ ]:
os.makedirs(os.path.dirname(RUN_DIR), exist_ok=True)
shutil.copytree(LOCAL_DIR, RUN_DIR, dirs_exist_ok=True)
u = metrics['usage']
cost = metrics['cost_usd_estimate']
cost_text = 'cost not token-priced (GPU time)' if cost is None else f'about ${cost:.2f}'  # None: an open model on the pod
per_call = sum(r['seconds'] for r in metrics['per_call']) / max(1, metrics['calls'])
print(f"OK: {SETTINGS['experiment']} {MODEL} effort {EFFORT}, commit {COMMIT[:7]}: {metrics['calls']} calls, "
      f"{metrics['format_failures']} failures, decisions agree {metrics['decisions_agreeing']}/{metrics['decisions_judged']}; "
      f"tokens in {u['input_tokens']}, cache write {u['cache_creation_input_tokens']}, cache read {u['cache_read_input_tokens']}, "
      f"out {u['output_tokens']}; cache {'ok' if cache_ok else 'NOT read'}; {cost_text}; {per_call:.0f} s per call. "
      f"Saved to runs/{SETTINGS['name']}/.")